# Single-Asset Quant Research / 单币量化研究

这个 notebook 用来做一件更基础但很重要的事：在研究 LP 区间之前，先把某个币本身的价格行为看清楚。

你只需要指定一个 `asset`，例如 `BTC`、`SUI`、`SOL`，它会：

- 从 Binance 路线获取价格数据
- 自动处理 direct / ratio / proxy 路径
- 计算一组常见的 quant diagnostics
- 画出趋势、波动、回撤、动量、分布与自相关图

研究说明文档见：`/Users/lilith/dev/web3/lpquant/docs/single-asset-quant-research.md`


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from app.research import (
    AssetDiagnosticsRequest,
    format_source,
    plot_asset_diagnostics_dashboard,
    rename_for_display,
    run_asset_diagnostics,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)


## 运行参数

这里最关键的是 `asset`。如果你把 `asset = "SUI"`，它就会优先尝试 `SUI/USDC`，如果 Binance 没有直连，就自动走 ratio 或 proxy 路径。

`interval` 是采样频率；`days` 是回看的历史长度。第一轮建议先用 `1d + 730 days`。


In [ ]:
asset = "BTC"
quote = "USDC"
interval = "1d"
days = 730

request = AssetDiagnosticsRequest(
    asset=asset,
    quote=quote,
    interval=interval,
    days=days,
)
result = run_asset_diagnostics(request)

display(
    Markdown(
        f"**Pair / 交易对:** {result.dataset.pair.symbol}  \n"
        f"**Sampling interval / 采样周期:** {result.dataset.interval}  \n"
        f"**Source / 数据源:** {format_source(result.dataset.source)}  \n"
        f"**Bars / 样本数:** {len(result.dataset.frame)}"
    )
)

if result.dataset.notes:
    display(Markdown("**Notes / 说明**\n" + "\n".join(f"- {note}" for note in result.dataset.notes)))


## 这套分析看哪些东西

这份模板把常见单币研究拆成几块：

- `Trend`：均线、Bollinger、MACD，帮助判断趋势和位置
- `Risk`：realized vol、downside vol、ATR、drawdown，帮助判断区间需要多宽
- `Momentum`：RSI 和 MACD hist，帮助看短中期动量是否一致
- `Distribution`：收益分布、VaR、CVaR，帮助看尾部风险
- `Serial dependence`：return autocorr 和 abs-return autocorr，帮助看趋势延续、反转和 volatility clustering

下面这张窗口表，会告诉你每个指标用了多长的 lookback。注意这里是用“天”定义研究窗口，再自动换算成对应的 bars。


In [ ]:
rename_for_display(result.window_config)


## Snapshot / 当前状态快照

这一块更像是“当前体检报告”。你可以先快速看：

- 当前是 `bull trend`、`bear trend` 还是 `range`
- 当前波动是在历史上偏高还是偏低
- 当前价格离中短期均线偏离了多少
- 当前 RSI、MACD hist、ATR、drawdown 在什么位置


In [ ]:
snapshot_columns = [
    "asset",
    "quote",
    "pair",
    "interval",
    "trend_regime",
    "vol_regime",
    "momentum_regime",
    "current_price",
    "cumulative_return_pct",
    "trend_vs_short_sma_pct",
    "trend_vs_medium_sma_pct",
    "realized_vol_pct",
    "downside_vol_pct",
    "atr_pct",
    "rsi",
    "macd_hist",
    "volume_zscore",
    "drawdown_pct",
]
rename_for_display(result.snapshot[snapshot_columns])


## 收益、风险与自相关

`annualized return / vol / Sharpe` 更像粗略摘要，不要把它们当成交易信号本身。

这部分更值得重点看的是：

- `max drawdown` 有多深
- `VaR` 和 `CVaR` 有多差
- `skewness` 和 `excess kurtosis` 是否说明分布偏斜、厚尾
- `abs return autocorr` 是否明显高于 `return autocorr`

如果绝对收益自相关很高，通常说明 volatility clustering 很明显，这对 LP 区间宽度选择很关键。


In [ ]:
display(rename_for_display(result.return_stats))
display(rename_for_display(result.horizon_returns))
display(rename_for_display(result.autocorrelation))


## Dashboard / 图形化诊断

这张图是最值得反复看的部分：

- 左上：价格、均线、Bollinger band
- 右上：drawdown
- 第二行：波动率与 volume
- 第三行：RSI 和 MACD
- 第四行：收益分布与自相关


In [ ]:
plot_asset_diagnostics_dashboard(
    result,
    title=f"{asset}/{quote} single-asset quant dashboard",
)


## 阅读建议与参考资料

推荐先按这个顺序理解：

1. 先看价格图、drawdown、rolling vol，理解这个币是 trend asset 还是 shock-prone asset。
2. 再看收益分布、VaR/CVaR、autocorrelation，理解这个币的尾部和 clustering 特征。
3. 最后再回到 LP 问题：如果这个币 trend 强、drawdown 深、vol clustering 明显，那区间就不能太窄，也不能太久不重设。

相关阅读：

- [Cont (2001), Empirical properties of asset returns: stylized facts and statistical issues](https://ideas.repec.org/a/taf/quantf/v1y2001i2p223-236.html)
- [Engle (2003), Risk and Volatility: Econometric Models and Financial Practice](https://www.nobelprize.org/prizes/economic-sciences/2003/engle/lecture/)
- [Brock, Lakonishok, and LeBaron (1992), Simple Technical Trading Rules and the Stochastic Properties of Stock Returns](https://econpapers.repec.org/RePEc:bla:jfinan:v:47:y:1992:i:5:p:1731-64)
- [Jegadeesh and Titman (1993), Returns to Buying Winners and Selling Losers](https://ideas.repec.org/a/bla/jfinan/v48y1993i1p65-91.html)
- [Liu and Tsyvinski (2018), Risks and Returns of Cryptocurrency](https://www.nber.org/papers/w24877)
- [Liu, Tsyvinski, and Wu (2019), Common Risk Factors in Cryptocurrency](https://www.nber.org/papers/w25882)

如果你下一步要把它和 LP 区间优化连起来，最自然的做法是：先把这里的 `trend / vol / drawdown / autocorr` 结果拿来决定，某个币更适合 `短 holding period + 窄区间`，还是 `长 holding period + 宽区间`。
